# 04 - Desemprego e Atividade Economica

## Objetivo
Analisar taxa de desemprego e variacao da producao industrial.

## Fluxo
1. Carregar dados
2. Analise de desemprego
3. Analise de producao industrial
4. Correlacao entre variaveis
5. Ciclos economicos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)

## 1. Carregar dados

In [ ]:
caminho_dados = Path('../dados/brutos/indicadores_consolidados.csv')
df = pd.read_csv(caminho_dados)
df['data'] = pd.to_datetime(df['data'])
df = df.sort_values('data').reset_index(drop=True)

print(f'Dados carregados: {len(df)} observacoes')

## 2. Analise de Taxa de Desemprego

In [ ]:
taxa_desemprego = df['taxa_desemprego']

print('TAXA DE DESEMPREGO:')
print('=' * 50)
print(f'Media: {taxa_desemprego.mean():.2f}%')
print(f'Mediana: {taxa_desemprego.median():.2f}%')
print(f'Desvio Padrao: {taxa_desemprego.std():.2f}%')
print(f'Minimo: {taxa_desemprego.min():.2f}% (data: {df.loc[taxa_desemprego.idxmin(), "data"].strftime("%b/%Y")})')
print(f'Maximo: {taxa_desemprego.max():.2f}% (data: {df.loc[taxa_desemprego.idxmax(), "data"].strftime("%b/%Y")})')
print()
print(f'Desemprego Atual: {taxa_desemprego.iloc[-1]:.2f}%')

## 3. Analise de Producao Industrial

In [ ]:
prod_industrial = df['variacao_producao_industrial']

print('VARIACAO DA PRODUCAO INDUSTRIAL:')
print('=' * 50)
print(f'Media: {prod_industrial.mean():.2f}%')
print(f'Mediana: {prod_industrial.median():.2f}%')
print(f'Desvio Padrao: {prod_industrial.std():.2f}%')
print(f'Minimo: {prod_industrial.min():.2f}% (data: {df.loc[prod_industrial.idxmin(), "data"].strftime("%b/%Y")})')
print(f'Maximo: {prod_industrial.max():.2f}% (data: {df.loc[prod_industrial.idxmax(), "data"].strftime("%b/%Y")})')
print()
print(f'Producao Atual: {prod_industrial.iloc[-1]:.2f}%')

## 4. Comparacao visual: Desemprego vs Producao

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Desemprego
ax1.plot(df['data'], df['taxa_desemprego'], linewidth=2.5, color='#d62728', marker='o', markersize=4)
ax1.fill_between(df['data'], df['taxa_desemprego'], alpha=0.3, color='#d62728')
ax1.axhline(y=df['taxa_desemprego'].mean(), color='black', linestyle='--', linewidth=2, label=f'Media: {df["taxa_desemprego"].mean():.2f}%')
ax1.set_title('Taxa de Desemprego (PNAD Continua)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Taxa (%)')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Producao Industrial
ax2.plot(df['data'], df['variacao_producao_industrial'], linewidth=2.5, color='#2ca02c', marker='s', markersize=4)
ax2.fill_between(df['data'], df['variacao_producao_industrial'], alpha=0.3, color='#2ca02c')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.5)
ax2.axhline(y=df['variacao_producao_industrial'].mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {df["variacao_producao_industrial"].mean():.2f}%')
ax2.set_title('Variacao da Producao Industrial', fontsize=14, fontweight='bold')
ax2.set_ylabel('Variacao (%)')
ax2.set_xlabel('Data')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

## 5. Correlacao com IPCA

In [ ]:
print('CORRELACAO COM INFLACAO (IPCA MENSAL):')
print('=' * 50)

corr_desemprego = df['ipca_mensal'].corr(df['taxa_desemprego'])
corr_producao = df['ipca_mensal'].corr(df['variacao_producao_industrial'])

print(f'IPCA vs Desemprego: {corr_desemprego:+.3f}')
print(f'  Interpretacao: {"Correlacao negativa" if corr_desemprego < 0 else "Correlacao positiva"}')
print()
print(f'IPCA vs Producao Industrial: {corr_producao:+.3f}')
print(f'  Interpretacao: {"Correlacao negativa" if corr_producao < 0 else "Correlacao positiva"}')

## 6. Scatter plot - Desemprego vs IPCA

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Scatter: Desemprego vs IPCA
ax1.scatter(df['taxa_desemprego'], df['ipca_mensal'], s=100, alpha=0.6, c=range(len(df)), cmap='viridis')
z = np.polyfit(df['taxa_desemprego'], df['ipca_mensal'], 1)
p = np.poly1d(z)
ax1.plot(df['taxa_desemprego'], p(df['taxa_desemprego']), "r--", alpha=0.8, linewidth=2)
ax1.set_xlabel('Taxa de Desemprego (%)')
ax1.set_ylabel('IPCA Mensal (%)')
ax1.set_title(f'IPCA vs Desemprego (Corr: {corr_desemprego:+.3f})', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Scatter: Producao vs IPCA
ax2.scatter(df['variacao_producao_industrial'], df['ipca_mensal'], s=100, alpha=0.6, c=range(len(df)), cmap='viridis')
z = np.polyfit(df['variacao_producao_industrial'], df['ipca_mensal'], 1)
p = np.poly1d(z)
ax2.plot(df['variacao_producao_industrial'], p(df['variacao_producao_industrial']), "r--", alpha=0.8, linewidth=2)
ax2.set_xlabel('Variacao Producao Industrial (%)')
ax2.set_ylabel('IPCA Mensal (%)')
ax2.set_title(f'IPCA vs Producao (Corr: {corr_producao:+.3f})', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Resumo da analise

In [ ]:
print('
RESUMO - DESEMPREGO E ATIVIDADE ECONOMICA:')
print('=' * 60)
print()
print('DESEMPREGO:')
print(f'  Media: {taxa_desemprego.mean():.2f}%')
print(f'  Variacao: {taxa_desemprego.min():.2f}% a {taxa_desemprego.max():.2f}%')
print(f'  Tendencia: {"Crescente" if taxa_desemprego.iloc[-1] > taxa_desemprego.mean() else "Decrescente"}')
print()
print('PRODUCAO INDUSTRIAL:')
print(f'  Media: {prod_industrial.mean():.2f}%')
print(f'  Variacao: {prod_industrial.min():.2f}% a {prod_industrial.max():.2f}%')
print()
print('CORRELACOES COM INFLACAO:')
print(f'  Desemprego-IPCA: {corr_desemprego:+.3f}')
print(f'  Producao-IPCA: {corr_producao:+.3f}')